In [ ]:
%load ../lib/RecursiveSet.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

# Identifying Tautologies

In this notebook we develop a function `tautology` that takes a formula $f$ from propositional logic and checks whether $f$ is a *tautology*.
As tautologies are propositional formulas, we first have to load the record classes representing formulas and the parser for propositional logic.

In [ ]:
%load Formula.jsh
%load PropositionalLogicParser.jsh

To be able to compute all propositional valuations for a given formula $f$ we first need to determine the set of all variables that occur in $f$.  The function $\texttt{collectVars}(f)$ takes a formula $f$ from propositional logic and computes all propositional variables occurring in $f$.  This function is defined recursively.

In [ ]:
RecursiveSet<String> collectVars(Formula f) {
    return switch (f) {
        case Var(var p)            -> set(p);
        case Verum _, Falsum _     -> set();
        case Not(var g)            -> collectVars(g);
        case And(var g, var h)     -> collectVars(g).union(collectVars(h));
        case Or(var g, var h)      -> collectVars(g).union(collectVars(h));
        case Implies(var g, var h) -> collectVars(g).union(collectVars(h));
        case Equiv(var g, var h)   -> collectVars(g).union(collectVars(h));
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

In the case `Verum _, Falsum _` the character `_` denotes an *unnamed variable*: This case matches if `f` is either an instance of `Verum` or an instance of `Falsum`, but there is no need to bind `f` to a new variable.

In [ ]:
collectVars(parse("(p → q) ∧ (¬p → r) → ⊥"))

We have discussed the function <tt>evaluate</tt> previously.  The call 
$\texttt{evaluate}(f, I)$ takes a propsitional formula $f$ and a propositional valuation $I$, where $I$ is represented as a set of propositional variables.  It evaluates $f$ given $I$.

In [ ]:
boolean evaluate(Formula f, RecursiveSet<String> I) {
    return switch (f) {
        case Var(var p)            -> I.contains(p);
        case Verum()               -> true;
        case Falsum()              -> false;
        case Not(var g)            -> !evaluate(g, I);
        case And(var g, var h)     -> evaluate(g, I) && evaluate(h, I);
        case Or(var g, var h)      -> evaluate(g, I) || evaluate(h, I);
        case Implies(var g, var h) -> !evaluate(g, I) || evaluate(h, I);
        case Equiv(var g, var h)   -> evaluate(g, I) == evaluate(h, I);
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

Now we are ready to define the function $\texttt{tautology}(f)$ that takes a propositional formula $f$ and checks whether $f$ is a tautology.  The result is an object of type `Optional`:
* If $f$ is a tautology, the function returns `Optional.empty()`, i.e. there is no counter example.
* Otherwise, a set of variables $I$ is returned such that $f$ evaluates to <tt>false</tt> if all variables in $I$ are <tt>true</tt>, while all variables not in $I$ are <tt>false</tt>.  This set is wrapped as `Optional.of(I)`.

In [ ]:
Optional<RecursiveSet<String>> tautology(Formula f) {
    var P = collectVars(f);
    for (var I : P.powerset()) {
        if (!evaluate(f, I)) {
            return Optional.of(I);
        }
    }
    return Optional.empty();
}

The function $\texttt{test}(s)$ takes a string $s$ that can be parsed as a propositional formula and checks whether this formula is a tautology.

In [ ]:
void test(String s) {
    var f              = parse(s);
    var counterExample = tautology(f);
    if (counterExample.isEmpty()) {
        System.out.println("The formula " + s + " is a tautology.");
    } else {
        var P = collectVars(f);
        System.out.println("The formula " + s + " is not a tautology.");
        System.out.println("Counter example:");
        for (var x : P) {
            if (counterExample.get().contains(x)) {
                System.out.println(x + " ↦ true");
            } else {
                System.out.println(x + " ↦ false");
            }
        }
    }
}

Let us run a few tests.

Let's check the rules of *DeMorgan*.

In [ ]:
test("¬(p ∨ q) ↔ ¬p ∧ ¬q");

In [ ]:
test("¬(p ∧ q) ↔ ¬p ∨ ¬q");

In [ ]:
test("(p → q) → (¬p → q) → q");

In [ ]:
test("(p → q) → (¬p → ¬q)");

In [ ]:
test("(p → q) → (¬q → ¬p)");

In [ ]:
test("¬p ↔ (p → ⊥)");